In [ ]:
# Task 1: Download and Load Data

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Download diabetes dataset
url = "https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/refs/heads/master/diabetes.csv"
df = pd.read_csv(url)

# Separate features (X) and target (y)
X = df.iloc[:, :-1].values  # First 8 columns: features
y = df.iloc[:, -1].values    # Last column: diabetes (0 or 1)

# Split into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")


In [ ]:
# Task 2: Neural Network Implementation

class DiabetesNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Architecture: 8 → 6 → 4 → 2
        self.backbone = nn.Sequential(
            nn.Linear(in_features=8, out_features=6),
            nn.ReLU(),
            nn.Linear(in_features=6, out_features=4),
            nn.ReLU(),
            nn.Linear(in_features=4, out_features=2),
        )
    
    def forward(self, x):
        return self.backbone(x)

model = DiabetesNet()
print(model)


In [ ]:
# Task 3: Neural Network Training

from torch.utils.data import TensorDataset, DataLoader

# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)  # CrossEntropyLoss requires Long type
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

# Create DataLoaders for batch processing
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Loss function and optimizer
loss_func = nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(), lr=0.01)

# Training configuration
total_epochs = 150

# Metrics storage
all_train_losses = []
all_train_acc = []
all_test_losses = []
all_test_acc = []

print("Starting training...")


In [ ]:
# Test function
def test(model, test_loader, loss_func):
    model.eval()
    losses = []
    preds = []
    gt = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            logits = model(inputs)
            loss = loss_func(logits, labels)
            losses.append(loss.item())
            pred = torch.argmax(logits, dim=1)
            preds.extend(pred.numpy())
            gt.extend(labels.numpy())
    
    test_acc = accuracy_score(gt, preds)
    test_loss = sum(losses) / len(losses)
    return test_loss, test_acc


In [ ]:
# Training loop
for epoch in range(total_epochs):
    model.train()
    losses = []
    preds = []
    gt = []
    
    for inputs, labels in train_loader:
        logits = model(inputs)
        loss = loss_func(logits, labels)
        
        # Backpropagation
        loss.backward()
        optim.step()
        optim.zero_grad()
        
        losses.append(loss.item())
        pred = torch.argmax(logits, dim=1)
        preds.extend(pred.detach().numpy())
        gt.extend(labels.detach().numpy())
    
    train_acc = accuracy_score(gt, preds)
    train_loss = sum(losses) / len(losses)
    all_train_acc.append(train_acc)
    all_train_losses.append(train_loss)
    
    # Test evaluation
    test_loss, test_acc = test(model, test_loader, loss_func)
    all_test_losses.append(test_loss)
    all_test_acc.append(test_acc)
    
    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1}/{total_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

print(f"\nFinal Test Accuracy: {max(all_test_acc)*100:.2f}%")


In [ ]:
# Visualize training results
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(all_train_losses, label='Train Loss', color='orange')
plt.plot(all_test_losses, label='Test Loss', color='blue')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Test Loss')

plt.subplot(1, 2, 2)
plt.plot(all_train_acc, label='Train Accuracy', color='orange')
plt.plot(all_test_acc, label='Test Accuracy', color='blue')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Test Accuracy')

plt.tight_layout()
plt.show()
